<a href="https://colab.research.google.com/github/benitezlucio/citi-bike-demand-intelligence/blob/main/notebooks/01_data_acquisition_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Citi Bike Demand Intelligence
## 01 — Adquisición y preparación de datos

Este notebook construye los datasets procesados utilizados por el proyecto a partir de dos fuentes públicas: los viajes históricos de Citi Bike y datos meteorológicos horarios de Open-Meteo.

El objetivo de esta etapa es obtener un pipeline **reproducible**, validar la calidad de los datos, aplicar las reglas de limpieza definidas, integrar ambas fuentes y exportar tablas compactas listas para el análisis exploratorio.

> El análisis de hipótesis, las visualizaciones y los insights se desarrollan en `02_eda_insights.ipynb`.

## Fuentes y período de análisis

### Citi Bike Trip History
Datos a nivel de viaje publicados por Citi Bike. Se utilizan, entre otras variables, fecha y hora de inicio/finalización, estaciones, coordenadas, tipo de bicicleta y categoría de usuario.

### Open-Meteo Historical Weather API
Datos meteorológicos horarios para Nueva York, incluyendo temperatura, sensación térmica, humedad, precipitación, nieve, velocidad del viento y código meteorológico.

### Período
Se analizan viajes iniciados entre el **1 de enero y el 31 de diciembre de 2024**.

La granularidad final principal será **una fila por hora**, adecuada para analizar demanda y para una futura etapa de Machine Learning orientada a predicción horaria.

In [ ]:
import subprocess
import zipfile
from pathlib import Path

import pandas as pd
import requests

In [ ]:
REPO_NAME = "citi-bike-demand-intelligence"
REPO_URL = "https://github.com/benitezlucio/citi-bike-demand-intelligence.git"


def find_project_root(start=None):
    """Busca hacia arriba la raíz del repositorio."""
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "README.md").exists()
            and (candidate / "notebooks").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    return None


# En local buscamos el repo desde el directorio actual.
PROJECT_ROOT = find_project_root()

# En Colab, si el repo no existe todavía, lo clonamos automáticamente.
if PROJECT_ROOT is None:
    try:
        import google.colab

        repo_path = Path("/content") / REPO_NAME

        if not repo_path.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(repo_path)],
                check=True
            )

        PROJECT_ROOT = repo_path

    except ImportError:
        raise FileNotFoundError(
            "No se pudo detectar la raíz del repositorio. "
            "Ejecutá el notebook desde dentro del repo clonado."
        )


DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

In [ ]:
# Configuración general del proyecto

YEAR = 2024
CHUNK_SIZE = 250_000

CITIBIKE_BASE_URL = "https://s3.amazonaws.com/tripdata"

NYC_LATITUDE = 40.7128
NYC_LONGITUDE = -74.0060
NYC_TIMEZONE = "America/New_York"

ANALYSIS_START = pd.Timestamp("2024-01-01 00:00:00")
ANALYSIS_END = pd.Timestamp("2025-01-01 00:00:00")

print(f"Período de análisis: {ANALYSIS_START} → {ANALYSIS_END}")
print(f"Tamaño de chunk: {CHUNK_SIZE:,}")

## Contrato de datos de Citi Bike

Los archivos históricos de Citi Bike contienen registros a nivel de viaje. Cada fila representa un viaje individual.

Se esperan 13 variables:

| Variable | Descripción |
|---|---|
| `ride_id` | Identificador del viaje |
| `rideable_type` | Tipo de bicicleta |
| `started_at` | Inicio del viaje |
| `ended_at` | Finalización del viaje |
| `start_station_name` | Estación de origen |
| `start_station_id` | ID de estación de origen |
| `end_station_name` | Estación de destino |
| `end_station_id` | ID de estación de destino |
| `start_lat` / `start_lng` | Coordenadas de inicio |
| `end_lat` / `end_lng` | Coordenadas de destino |
| `member_casual` | Tipo de usuario |

Validar este esquema permite detectar cambios inesperados en la fuente antes de procesar millones de registros.

In [ ]:
EXPECTED_CITIBIKE_COLUMNS = [
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual",
]


def validate_columns(df):
    missing = set(EXPECTED_CITIBIKE_COLUMNS) - set(df.columns)

    if missing:
        raise ValueError(
            f"Faltan columnas esperadas de Citi Bike: {sorted(missing)}"
        )

## Adquisición de Citi Bike

Los ZIP mensuales se descargan directamente desde el bucket público de Citi Bike.

Como algunos meses pueden estar publicados con nombres distintos y cada ZIP puede contener más de un CSV, la función contempla ambas situaciones. Los archivos crudos se guardan en `data/raw/`, carpeta que no se versiona en Git.

In [ ]:
def download_citibike_month(year, month, raw_dir=RAW_DIR):
    """Descarga el ZIP mensual de Citi Bike si todavía no existe localmente."""
    month_key = f"{year}{month:02d}"

    candidates = [
        f"{month_key}-citibike-tripdata.csv.zip",
        f"{month_key}-citibike-tripdata.zip",
    ]

    for filename in candidates:
        output_path = raw_dir / filename

        # Evitamos volver a descargar archivos que ya están disponibles.
        if output_path.exists():
            print(f"Already downloaded: {filename}")
            return output_path

        url = f"{CITIBIKE_BASE_URL}/{filename}"
        print(f"Trying: {url}")

        response = requests.get(url, stream=True, timeout=60)

        if response.status_code == 200:
            with open(output_path, "wb") as file_obj:
                for block in response.iter_content(chunk_size=1024 * 1024):
                    if block:
                        file_obj.write(block)

            print(f"Downloaded: {output_path.name}")
            return output_path

    raise FileNotFoundError(
        f"No se encontró un archivo de Citi Bike para {year}-{month:02d}"
    )

## Diagnóstico inicial de calidad — enero 2024

Antes de escalar el procesamiento a todo el año se utiliza enero como mes de diagnóstico.

La idea no es cargar el mes completo en memoria: se recorre por bloques para revisar nulos, categorías, duplicados de `ride_id`, duraciones inválidas y registros fuera del período definido.

In [ ]:
def profile_citibike_month(zip_path):
    """Perfila un mes por chunks sin conservar el dataset completo en memoria."""
    total_rows = 0
    null_counts = None
    unique_values = {
        "rideable_type": set(),
        "member_casual": set(),
    }

    seen_ride_ids = set()
    duplicate_ride_ids = 0
    invalid_duration_rows = 0
    out_of_period_rows = 0
    duration_parts = []

    with zipfile.ZipFile(zip_path) as zip_ref:
        csv_files = sorted(
            name for name in zip_ref.namelist()
            if name.lower().endswith(".csv")
        )

        for csv_name in csv_files:
            print(f"Procesando: {csv_name}")

            for chunk in pd.read_csv(
                zip_ref.open(csv_name),
                chunksize=CHUNK_SIZE,
                low_memory=False
            ):
                validate_columns(chunk)

                total_rows += len(chunk)

                # Sumamos nulos sin conservar todos los registros.
                chunk_nulls = chunk.isna().sum()
                null_counts = (
                    chunk_nulls
                    if null_counts is None
                    else null_counts.add(chunk_nulls, fill_value=0)
                )

                for column in unique_values:
                    unique_values[column].update(
                        chunk[column].dropna().unique()
                    )

                started = pd.to_datetime(chunk["started_at"], errors="coerce")
                ended = pd.to_datetime(chunk["ended_at"], errors="coerce")
                duration = (ended - started).dt.total_seconds() / 60

                duration_parts.append(duration.dropna())

                invalid_duration_rows += (
                    started.isna()
                    | ended.isna()
                    | (ended <= started)
                ).sum()

                in_period = (
                    (started >= ANALYSIS_START)
                    & (started < ANALYSIS_END)
                )
                out_of_period_rows += (~in_period).sum()

                # Controlamos IDs repetidos también entre distintos chunks de enero.
                ids = chunk["ride_id"].dropna()
                duplicate_ride_ids += ids.duplicated().sum()

                unique_ids = ids.drop_duplicates()
                duplicate_ride_ids += unique_ids.isin(seen_ride_ids).sum()
                seen_ride_ids.update(unique_ids.tolist())

    durations = pd.concat(duration_parts, ignore_index=True)

    duration_summary = durations.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )

    return {
        "total_rows": total_rows,
        "null_counts": null_counts.astype("int64"),
        "unique_values": unique_values,
        "duplicate_ride_ids": int(duplicate_ride_ids),
        "invalid_duration_rows": int(invalid_duration_rows),
        "out_of_period_rows": int(out_of_period_rows),
        "duration_summary": duration_summary,
    }

In [ ]:
january_zip = download_citibike_month(YEAR, 1)

with zipfile.ZipFile(january_zip) as zip_ref:
    january_csv_files = sorted(
        name for name in zip_ref.namelist()
        if name.lower().endswith(".csv")
    )

print("Archivo:", january_zip.name)
print("CSV dentro del ZIP:", len(january_csv_files))
for name in january_csv_files:
    print("-", name)

january_profile = profile_citibike_month(january_zip)

In [ ]:
null_summary = pd.DataFrame({
    "null_count": january_profile["null_counts"],
    "null_percentage": (
        january_profile["null_counts"]
        / january_profile["total_rows"]
        * 100
    ).round(2),
})

print(f"Registros: {january_profile['total_rows']:,}")
print(f"Columnas esperadas: {len(EXPECTED_CITIBIKE_COLUMNS)}")
print(f"Ride IDs duplicados: {january_profile['duplicate_ride_ids']:,}")
print(f"Duraciones inválidas: {january_profile['invalid_duration_rows']:,}")
print(f"Registros fuera de 2024: {january_profile['out_of_period_rows']:,}")
print("Categorías:", january_profile["unique_values"])

display(null_summary)
display(january_profile["duration_summary"].to_frame("duration_minutes"))

## Criterios de calidad y limpieza

El diagnóstico de enero muestra una buena calidad general de los datos. Los nulos se concentran principalmente en las variables de estaciones y coordenadas, mientras que las variables centrales para analizar demanda no presentan faltantes relevantes.

Se adoptan las siguientes reglas:

- Se conservan únicamente viajes iniciados dentro de 2024.
- Los viajes con `ended_at <= started_at` se consideran temporalmente inválidos.
- El diagnóstico de enero no detectó `ride_id` duplicados; durante el procesamiento anual se eliminan duplicados evidentes dentro de cada chunk como salvaguarda.
- Los registros sin estación se conservan para análisis temporal, climático y por tipo de usuario.
- Los registros sin estación de origen se excluyen únicamente del análisis de concentración por estación.
- No se eliminan viajes largos sólo por su duración: el objetivo principal es estudiar cantidad de viajes iniciados, no duración.
- No se aplican filtros de outliers sin una justificación analítica.

## Procesamiento anual por chunks

Los aproximadamente 44 millones de viajes de 2024 no se cargan simultáneamente en memoria.

Cada CSV se procesa en bloques de 250.000 filas y se transforma inmediatamente en tres agregados:

1. demanda total por hora;
2. demanda horaria por tipo de usuario y bicicleta;
3. viajes iniciados por estación.

Este enfoque reduce considerablemente el uso de memoria y mantiene la reproducción posible en equipos convencionales.

In [ ]:
def aggregate_citibike_zip(zip_path):
    """
    Procesa todos los CSV de un ZIP por bloques y devuelve:
    - demanda horaria total,
    - demanda horaria segmentada,
    - demanda por estación de origen.
    """
    hourly_parts = []
    segmented_parts = []
    station_parts = []

    with zipfile.ZipFile(zip_path) as zip_ref:
        csv_files = sorted(
            name for name in zip_ref.namelist()
            if name.lower().endswith(".csv")
        )

        for csv_name in csv_files:
            print(f"Processing: {csv_name}")

            for chunk in pd.read_csv(
                zip_ref.open(csv_name),
                chunksize=CHUNK_SIZE,
                low_memory=False
            ):
                validate_columns(chunk)

                # Salvaguarda simple frente a duplicados dentro del mismo bloque.
                chunk = chunk.drop_duplicates(subset="ride_id")

                chunk["started_at"] = pd.to_datetime(
                    chunk["started_at"],
                    errors="coerce"
                )
                chunk["ended_at"] = pd.to_datetime(
                    chunk["ended_at"],
                    errors="coerce"
                )

                # Filtramos estrictamente por fecha de inicio del viaje.
                chunk = chunk.loc[
                    (chunk["started_at"] >= ANALYSIS_START)
                    & (chunk["started_at"] < ANALYSIS_END)
                ].copy()

                # Eliminamos únicamente registros temporalmente inválidos.
                chunk = chunk.loc[
                    chunk["ended_at"].notna()
                    & (chunk["ended_at"] > chunk["started_at"])
                ].copy()

                # Redondeamos el inicio a la hora para construir la demanda horaria.
                chunk["datetime"] = chunk["started_at"].dt.floor("h")

                hourly_parts.append(
                    chunk.groupby("datetime")
                    .size()
                    .rename("ride_count")
                    .reset_index()
                )

                segmented_parts.append(
                    chunk.groupby(
                        ["datetime", "member_casual", "rideable_type"],
                        dropna=False
                    )
                    .size()
                    .rename("ride_count")
                    .reset_index()
                )

                # Para H6 sólo necesitamos estaciones de origen válidas.
                valid_stations = chunk.dropna(
                    subset=["start_station_id", "start_station_name"]
                )

                station_parts.append(
                    valid_stations.groupby(
                        ["start_station_id", "start_station_name"]
                    )
                    .size()
                    .rename("ride_count")
                    .reset_index()
                )

    hourly = (
        pd.concat(hourly_parts, ignore_index=True)
        .groupby("datetime", as_index=False)["ride_count"]
        .sum()
    )

    segmented = (
        pd.concat(segmented_parts, ignore_index=True)
        .groupby(
            ["datetime", "member_casual", "rideable_type"],
            as_index=False
        )["ride_count"]
        .sum()
    )

    stations = (
        pd.concat(station_parts, ignore_index=True)
        .groupby(
            ["start_station_id", "start_station_name"],
            as_index=False
        )["ride_count"]
        .sum()
    )

    return hourly, segmented, stations

In [ ]:
monthly_hourly = []
monthly_segmented = []
monthly_stations = []
monthly_summary = []

for month in range(1, 13):
    print(f"\n{'=' * 50}")
    print(f"Processing {YEAR}-{month:02d}")
    print("=" * 50)

    zip_path = download_citibike_month(YEAR, month)
    hourly, segmented, stations = aggregate_citibike_zip(zip_path)

    monthly_hourly.append(hourly)
    monthly_segmented.append(segmented)
    monthly_stations.append(stations)

    monthly_summary.append({
        "month": month,
        "ride_count": hourly["ride_count"].sum(),
    })

print("\nAnnual monthly processing completed.")

In [ ]:
# Consolidamos los agregados mensuales en tres tablas anuales.

citibike_demand_hourly = (
    pd.concat(monthly_hourly, ignore_index=True)
    .groupby("datetime", as_index=False)["ride_count"]
    .sum()
    .sort_values("datetime")
    .reset_index(drop=True)
)

citibike_segmented_hourly = (
    pd.concat(monthly_segmented, ignore_index=True)
    .groupby(
        ["datetime", "member_casual", "rideable_type"],
        as_index=False
    )["ride_count"]
    .sum()
    .sort_values(["datetime", "member_casual", "rideable_type"])
    .reset_index(drop=True)
)

citibike_stations = (
    pd.concat(monthly_stations, ignore_index=True)
    .groupby(
        ["start_station_id", "start_station_name"],
        as_index=False
    )["ride_count"]
    .sum()
    .sort_values("ride_count", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
monthly_summary_df = pd.DataFrame(monthly_summary)
display(monthly_summary_df)

print("\nAnnual validation")
print("-" * 35)
print("Hourly timestamps:", f"{len(citibike_demand_hourly):,}")
print("Hourly rides:", f"{citibike_demand_hourly['ride_count'].sum():,}")
print("Segmented rides:", f"{citibike_segmented_hourly['ride_count'].sum():,}")
print("Station rides:", f"{citibike_stations['ride_count'].sum():,}")
print("First timestamp:", citibike_demand_hourly["datetime"].min())
print("Last timestamp:", citibike_demand_hourly["datetime"].max())

## Validación temporal

Durante la adquisición se detectaron **410 viajes iniciados el 31/12/2023** dentro de los archivos asociados a enero de 2024. Se excluyen porque están fuera del período definido.

Además, 2024 presenta una particularidad por Daylight Saving Time (DST):

- el **10 de marzo a las 02:00** esa hora local no existió en Nueva York;
- el **3 de noviembre a la 01:00** la hora ocurrió dos veces.

No se crea artificialmente la hora inexistente ni se asigna demanda cero. La hora repetida queda agregada bajo la misma etiqueta local en Citi Bike, ya que sus timestamps no incluyen el offset.

In [ ]:
expected_local_hours = pd.date_range(
    "2024-01-01 00:00:00",
    "2024-12-31 23:00:00",
    freq="h"
)

missing_citibike_hours = expected_local_hours.difference(
    citibike_demand_hourly["datetime"].unique()
)

print("Horas Citi Bike:", citibike_demand_hourly["datetime"].nunique())
print("Horas locales ausentes:")
print(missing_citibike_hours)

## Datos meteorológicos — Open-Meteo

Para integrar correctamente el clima con Citi Bike se solicitan timestamps Unix y luego se convierten a `America/New_York`.

Se pide un día extra a cada lado del período y se filtra **después** de la conversión. Esto permite representar correctamente los cambios de DST y evitar perder horas en los extremos del año.

In [ ]:
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

WEATHER_VARIABLES = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "snowfall",
    "wind_speed_10m",
    "weather_code",
]

params = {
    "latitude": NYC_LATITUDE,
    "longitude": NYC_LONGITUDE,
    "start_date": "2023-12-31",
    "end_date": "2025-01-01",
    "hourly": ",".join(WEATHER_VARIABLES),
    "timezone": NYC_TIMEZONE,
    "timeformat": "unixtime",
}

response = requests.get(OPEN_METEO_URL, params=params, timeout=60)
response.raise_for_status()

weather_data = response.json()
weather_raw = pd.DataFrame(weather_data["hourly"])

weather_raw["datetime_utc"] = pd.to_datetime(
    weather_raw["time"],
    unit="s",
    utc=True
)

weather_raw["datetime_local"] = (
    weather_raw["datetime_utc"]
    .dt.tz_convert(NYC_TIMEZONE)
)

# Quitamos el timezone sólo después de conservar la hora local correcta.
weather_raw["datetime"] = (
    weather_raw["datetime_local"]
    .dt.tz_localize(None)
)

In [ ]:
start_local = pd.Timestamp("2024-01-01 00:00:00", tz=NYC_TIMEZONE)
end_local = pd.Timestamp("2025-01-01 00:00:00", tz=NYC_TIMEZONE)

weather_2024 = weather_raw.loc[
    (weather_raw["datetime_local"] >= start_local)
    & (weather_raw["datetime_local"] < end_local)
].copy()

weather_2024.reset_index(drop=True, inplace=True)

duplicate_hours = weather_2024.loc[
    weather_2024["datetime"].duplicated(keep=False),
    ["datetime_local", "datetime"]
]

missing_weather_hours = expected_local_hours.difference(
    weather_2024["datetime"].unique()
)

print("Observaciones meteorológicas reales:", len(weather_2024))
print("Horas locales únicas:", weather_2024["datetime"].nunique())
print("Horas locales duplicadas:", weather_2024["datetime"].duplicated().sum())
print("Nulos meteorológicos:", int(weather_2024[WEATHER_VARIABLES].isna().sum().sum()))

print("\nHora repetida por DST:")
display(duplicate_hours)

print("Hora local inexistente por DST:")
print(missing_weather_hours)

In [ ]:
def mode_or_first(series):
    mode = series.mode()
    return mode.iloc[0] if not mode.empty else series.iloc[0]


# Citi Bike sólo conserva una etiqueta para la hora repetida.
# Consolidamos el clima a la misma granularidad antes del merge.
weather = (
    weather_2024
    .groupby("datetime", as_index=False)
    .agg(
        temperature_2m=("temperature_2m", "mean"),
        apparent_temperature=("apparent_temperature", "mean"),
        relative_humidity_2m=("relative_humidity_2m", "mean"),
        precipitation=("precipitation", "sum"),
        snowfall=("snowfall", "sum"),
        wind_speed_10m=("wind_speed_10m", "mean"),
        weather_code=("weather_code", mode_or_first),
        observation_hours=("datetime_utc", "count"),
    )
)

print("Weather rows:", len(weather))
print("Unique datetimes:", weather["datetime"].nunique())
print("Duplicated datetimes:", weather["datetime"].duplicated().sum())
print("\nObservation hours:")
print(weather["observation_hours"].value_counts().sort_index())

## Integración de demanda y clima

Antes del merge se comprueba que Citi Bike y Open-Meteo tengan exactamente el mismo conjunto de timestamps locales.

La integración se valida como `one_to_one`: una hora de demanda debe corresponder a una única fila meteorológica.

In [ ]:
citibike_times = set(citibike_demand_hourly["datetime"])
weather_times = set(weather["datetime"])

missing_weather = sorted(citibike_times - weather_times)
extra_weather = sorted(weather_times - citibike_times)

print("Citi Bike timestamps:", len(citibike_times))
print("Weather timestamps:", len(weather_times))
print("Missing weather timestamps:", len(missing_weather))
print("Extra weather timestamps:", len(extra_weather))

assert not missing_weather
assert not extra_weather

analysis_df = citibike_demand_hourly.merge(
    weather,
    on="datetime",
    how="left",
    validate="one_to_one"
)

print("\nMerged shape:", analysis_df.shape)
print("Total rides:", f"{analysis_df['ride_count'].sum():,}")
print("Null values after merge:", int(analysis_df.isna().sum().sum()))

## Ingeniería de variables

A partir de `datetime` se crean variables temporales útiles para el EDA y la futura etapa predictiva.

No se crea una variable `rush_hour`: hacerlo antes de analizar H1 introduciría la hipótesis dentro de los datos. Las franjas horarias utilizadas son descriptivas y más amplias.

In [ ]:
def add_temporal_features(df):
    df = df.copy()

    df["date"] = df["datetime"].dt.date
    df["hour"] = df["datetime"].dt.hour

    df["day_of_week_num"] = df["datetime"].dt.dayofweek
    df["day_of_week"] = df["datetime"].dt.day_name()

    df["month_num"] = df["datetime"].dt.month
    df["month"] = df["datetime"].dt.month_name()

    df["is_weekend"] = df["day_of_week_num"].isin([5, 6])

    season_map = {
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Autumn", 10: "Autumn", 11: "Autumn",
    }

    df["season"] = df["month_num"].map(season_map)

    def get_time_period(hour):
        if 0 <= hour <= 5:
            return "Early morning"
        elif 6 <= hour <= 9:
            return "Morning"
        elif 10 <= hour <= 15:
            return "Midday"
        elif 16 <= hour <= 19:
            return "Evening"
        else:
            return "Night"

    df["time_period"] = df["hour"].map(get_time_period)

    return df

In [ ]:
analysis_df = add_temporal_features(analysis_df)
citibike_segmented_hourly = add_temporal_features(
    citibike_segmented_hourly
)

# Indicadores simples para comparar horas con y sin precipitación/nieve.
analysis_df["has_precipitation"] = analysis_df["precipitation"] > 0
analysis_df["has_snowfall"] = analysis_df["snowfall"] > 0

## Validaciones finales

Antes de exportar se comprueba que las tablas mantengan los totales esperados, que la tabla principal tenga una fila por timestamp y que no existan valores faltantes después de la integración.

In [ ]:
print("FINAL VALIDATION")
print("=" * 45)

assert len(analysis_df) == 8_783
assert analysis_df["datetime"].nunique() == 8_783
assert analysis_df["datetime"].duplicated().sum() == 0
assert analysis_df["ride_count"].sum() == 44_302_480
assert analysis_df.isna().sum().sum() == 0

assert citibike_segmented_hourly["ride_count"].sum() == 44_302_480
assert citibike_stations["ride_count"].sum() == 44_272_646

assert len(weather) == 8_783
assert weather["observation_hours"].sum() == 8_784

print("analysis_df:", analysis_df.shape)
print("Total rides:", f"{analysis_df['ride_count'].sum():,}")
print("Segmented rides:", f"{citibike_segmented_hourly['ride_count'].sum():,}")
print("Station rides:", f"{citibike_stations['ride_count'].sum():,}")
print(
    "Rides excluded from station analysis:",
    f"{44_302_480 - citibike_stations['ride_count'].sum():,}"
)

print("\nAll final validations passed.")

## Exportación de datasets procesados

Se versionan únicamente tablas procesadas y compactas. Los ZIP crudos permanecen fuera de Git.

Los tres archivos permiten dos niveles de reproducibilidad:

- **Quick Start:** clonar el repo y comenzar directamente desde `02_eda_insights.ipynb`.
- **Full Reproduction:** ejecutar este notebook desde cero y regenerar los tres archivos.

In [ ]:
analysis_path = PROCESSED_DIR / "citibike_analysis_hourly_2024.csv"
segmented_path = PROCESSED_DIR / "citibike_segmented_hourly_2024.csv"
stations_path = PROCESSED_DIR / "citibike_stations_2024.csv"

analysis_df.to_csv(analysis_path, index=False)
citibike_segmented_hourly.to_csv(segmented_path, index=False)
citibike_stations.to_csv(stations_path, index=False)

for path in [analysis_path, segmented_path, stations_path]:
    print(
        path.name,
        "-",
        round(path.stat().st_size / 1024**2, 2),
        "MB"
    )

## Resultado de la preparación

El pipeline genera:

- **`citibike_analysis_hourly_2024.csv`** — una fila por hora con demanda, clima y features temporales.
- **`citibike_segmented_hourly_2024.csv`** — demanda horaria segmentada por `member_casual` y `rideable_type`.
- **`citibike_stations_2024.csv`** — viajes iniciados por estación de origen; los registros sin estación de inicio se excluyen únicamente de esta tabla.

La preparación queda separada del EDA para evitar un notebook único excesivamente largo. El análisis de H1–H6 continúa en `02_eda_insights.ipynb`.